# Date extraction

**Difficulty:** Beginner

Read a date's *parts* off a document with `Choice` questions — which month, which day, which year, or none at all — and let code do the calendar math to resolve them. The model never adds numbers; it only reads what the text actually says.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/date_extraction_cookbook (click to expand)</summary>

# Date extraction

> Extracts absolute and relative dates by asking TypeSafe for the parts named in a document, then resolving and validating them in code with confidence-based review.

`extract_date(document, role)` takes a document and a phrase naming the date you want (e.g. "the deadline to return the form"), and hands back a `date` with a confidence — flagging a low-confidence read, or one whose parts don't add up to a date at all (including a date the document never states). The date can be spelled out ("August 14, 2027") or relative to today ("tomorrow", "next Thursday").

## The questions: read the pieces, never the math

Seven `Choice` questions in one call. `mode` says how the date is written: `absolute` (names a month), `relative` (relative to today), or `none` (not stated at all). The other six read the pieces `mode` calls for — an absolute date needs `month`, `day`, `year`; a relative one needs `day_anchor` (today/tomorrow/day-after/a weekday), and a named weekday needs `weekday` and `week_offset` (this week vs next).

`year` lists one option per year in a fixed window, plus two escapes: `none` (the text states no year — code infers one) and `out_of_range` (a year is stated but isn't in the list — code flags it rather than guessing).

## Resolving in code: math, not model

`assemble()` turns the `Choice` answers into a concrete `date`, counting from a pinned `TODAY` so relative dates resolve the same way on every run:

- If `year` is `none`, infer the current year — bumped to next year if the resulting date is more than a month in the past (a January date mentioned in December almost always means next January).
- If `year` is `out_of_range`, flag it — don't guess a year outside the list.
- A named weekday with no qualifier means the next occurrence on or after today; `next` means the following calendar week; `current` means this week.
- **Confidence is the minimum across every part the resolved shape actually used** — one weak part is enough to send the whole date to review, even if the others were confident.

## Worked results

Four short documents, each fed to `extract_date`. Absolute dates with a stated year resolve cleanly at high confidence. A relative date ("next Thursday") resolves against the pinned reference date. A document where the phrase names something that isn't actually a date at all correctly comes back `mode: none` — nothing to resolve, flagged for review rather than guessed.

> The full cookbook also handles relative weekdays end-to-end (`day_anchor`, `weekday`, `week_offset`) and runs across four real documents with a fixed reference date for reproducibility — trimmed here to absolute dates, since that's the core mechanism (read parts, resolve in code, gate on the weakest part's confidence). Read the full cookbook at [docs.typesafe.ai/cookbooks/date_extraction_cookbook](https://docs.typesafe.ai/cookbooks/date_extraction_cookbook) any time.

</details>

## The core idea

Structured extraction as several narrow `Choice` questions instead of one open-ended "what's the date" question — mode, month, day, year, each with a closed set of options (lesson 2's Choice) so the model can never return something that isn't a valid month name or a real day-of-month number. The actual arithmetic (does March + 31 make sense? what year does a bare month imply?) is ordinary code, never asked of the model — exactly the "code owns deterministic work, System One handles narrow judgments" split from lesson 1. And the confidence-as-minimum trick is the same one SDE cascade used: one weak part is enough to distrust the whole result, no matter how confident the rest were.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";

const MONTHS: Record<string, number> = {
  January: 1, February: 2, March: 3, April: 4, May: 5, June: 6,
  July: 7, August: 8, September: 9, October: 10, November: 11, December: 12,
};
const YEARS = Array.from({ length: 21 }, (_, i) => 2020 + i); // a smaller window than the real 1900-2050
const REVIEW_BELOW = 0.6;

function dateQuestions(role: string) {
  const absent = "The document does not state this, or it is not this kind of date.";
  return {
    mode: choice(
      `How is ${role} written? 'absolute' = a calendar date naming a month; 'relative' = given relative to today; 'none' = the document does not state this date.`,
      { absolute: null, relative: null, none: null },
    ),
    month: choice(
      `If ${role} is an absolute calendar date, which month is it in?`,
      { ...Object.fromEntries(Object.keys(MONTHS).map((m) => [m, null])), none: absent },
    ),
    day: choice(
      `If ${role} is an absolute calendar date, which day of the month (1-31)?`,
      {
        ...Object.fromEntries(Array.from({ length: 31 }, (_, i) => [String(i + 1), null])),
        none: absent,
      },
    ),
    year: choice(
      `If ${role} is an absolute calendar date, which year? Pick 'none' if no year is stated (code infers it), or 'out_of_range' if a year is stated but not in the list.`,
      {
        ...Object.fromEntries(YEARS.map((y) => [String(y), null])),
        out_of_range: "A year is stated for this date but is outside the listed range.",
        none: "No year is stated for this date.",
      },
    ),
  };
}

interface Resolved {
  date: string | null;
  confidence: number | null;
  needsReview: boolean;
  note: string;
}

// Format from local getters, not toISOString() - toISOString() converts to UTC first,
// which silently shifts the date by a day for anyone east of UTC (e.g. new Date(2026, 2, 3)
// is local midnight, but its UTC ISO string can read 2026-03-02).
function formatLocalDate(d: Date): string {
  const year = d.getFullYear();
  const month = String(d.getMonth() + 1).padStart(2, "0");
  const day = String(d.getDate()).padStart(2, "0");
  return `${year}-${month}-${day}`;
}

function assemble(
  answers: Record<string, { choice: string; confidence: number }>,
  today: Date,
): Resolved {
  const confs: number[] = [answers.mode.confidence];
  const mode = answers.mode.choice;

  function result(resolved: string | null, note: string): Resolved {
    const confidence = confs.length ? Math.min(...confs) : null;
    const needsReview = resolved === null || confidence === null || confidence < REVIEW_BELOW;
    return { date: resolved, confidence, needsReview, note };
  }

  if (mode === "none") return result(null, "no such date stated");
  if (mode !== "absolute") return result(null, `mode '${mode}' not handled in this demo`);

  const month = answers.month.choice;
  const day = answers.day.choice;
  const year = answers.year.choice;
  confs.push(answers.month.confidence, answers.day.confidence, answers.year.confidence);

  if (month === "none" || day === "none" || !(month in MONTHS)) {
    return result(null, "absolute date incomplete");
  }
  if (year === "out_of_range") {
    return result(null, `year outside ${YEARS[0]}-${YEARS[YEARS.length - 1]}`);
  }
  const resolvedYear = year === "none" ? today.getFullYear() : Number(year);
  const monthNum = MONTHS[month];
  const dayNum = Number(day);
  const resolved = new Date(resolvedYear, monthNum - 1, dayNum);
  if (resolved.getMonth() + 1 !== monthNum || resolved.getDate() !== dayNum) {
    return result(null, `impossible date: ${month} ${day}, ${resolvedYear}`);
  }
  return result(formatLocalDate(resolved), "");
}

async function extractDate(document: string, role: string, today: Date) {
  const response = await client.systemOne({ state: document, questions: dateQuestions(role) });
  const answers = Object.fromEntries(
    Object.entries(response.answers).map((
      [k, v],
    ) => [k, { choice: v.choice, confidence: v.confidence }]),
  );
  return assemble(answers, today);
}

const TODAY = new Date(2026, 6, 30); // a fixed reference date, so results are reproducible

const documents: Record<string, string> = {
  contract:
    "This agreement is effective as of March 3, 2026, and the initial term ends on the 3rd of March, 2027.",
  invoice: "Payment is due 30 days from the invoice date of June 15.",
};

for (const [name, doc] of Object.entries(documents)) {
  const result = await extractDate(doc, "the date this document takes effect", TODAY);
  console.log(name, result);
}

## What just happened

Two documents, one shared question — "the date this document takes effect." The contract states a year explicitly ("March 3, 2026") and should resolve cleanly at high confidence. The invoice names a month and day ("June 15") but no year for the effective date itself — expect `year: "none"` and code inferring the year from `TODAY`.

**Try it yourself:** add a third document whose stated year is centuries away (e.g. "effective January 1, 1850") and predict whether `year` comes back `out_of_range` — check that the result is `null` with `needsReview: true` rather than a guess.

## Related

- `lessons/core-concepts/0002-choice.ipynb` — the `Choice` primitive, here used to read structured pieces instead of picking a single category
- `cookbooks/extraction/sde-cascade.ipynb` — the same "confidence as the minimum across parts used" trick, applied to gate an escalation instead of a review flag
- `cookbooks/extraction/pre-parsed-value-extraction.ipynb` — a related extraction technique: code finds candidates first, the model only picks among them

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.